## Discrete Multivariate Distributions

Multivariate distributions describe the probabilistic behavior of *multiple* quantities and their relationships. Every Statistical and Machine Learning method involves multiple random variables (RVs) and their distributions: joint, conditional, and marginal. In fact, the foundation of each method lies in how it models the dependence structure of the RVs, which is typically done in a way that facilitates analysis and/or computation. In other words, different models make assumptions about how the variables are related to allow for efficient processing. Experienced Data Scientists understand the modelling assumptions and their limitations, allowing them to choose and improve appropriate methods for the problem at hand.

In this notebook we build a [Naive Bayes Classifier (NBC)](https://en.wikipedia.org/wiki/Naive_Bayes_classifier) for spam detection, one of the earliest and most widespread applications of [Statistical classification](https://en.wikipedia.org/wiki/Statistical_classification). Although the probability model for NBC is very simple, the method works extremely well for email filtering and other text classification. We will first look at the model details, then implement the NBC in Python and compare its results with the built-in function in [scikit-learn](https://scikit-learn.org/stable/), Python's principal Machine Learning library.

### Naive Bayes Classification

Classification is the problem of *predicting the class* of an object based on its other observed characteristics, which are probabilistically related to its class. We denote the class by the *categorical* (i.e., discrete finite) RV $Y$, called the *response* or *label*, and we denote the related variables by $X_1,\ldots,X_p$, called the *predictors* or *features*. For our purposes we will assume the predictors are also discrete finite RVs. To summarize, the goal of NBC is to predict the (unknown) value of $Y$ based on the (observed) values of $X_1,\ldots, X_p$.

##### Maximum A Posteriori Classification

Assume for starters that we know the *joint* distribution of all $(p+1)$ RVs $Y,X_1,\ldots, X_p$. If we observed the values $x_1,\ldots, x_p$ for $X_1,\ldots, X_p$, our best description of $Y$ would be the *conditional* distribution:
$$ P( Y = y | X_1 =x_1, \ldots, X_p = x_p ),\quad \text{ for }  y \in \text{ range }(Y) $$
which gives the conditional probabilities of the possible values of $Y$. If we then had to *guess* which value the RV $Y$ takes, we would choose the *most likely* one, i.e. the one that maximizes the conditional probability:
$$ \hat{y} =  \arg \max_y P( Y = y | X_1 =x_1, \ldots, X_p = x_p ) $$
This approach is called [Maximum a Posteriori (MAP)](https://en.wikipedia.org/wiki/Maximum_a_posteriori_estimation) prediction, from the name of the conditional distribution, which is called the [*posterior*](https://en.wikipedia.org/wiki/Posterior_probability) in this context.

Furthermore, from Bayes' theorem we get:
$$ P(Y=y|X_1=x_1,\ldots,X_p=x_p) = \frac{  P(Y=y, X_1 = x_1, \ldots, X_p = x_p ) } { P( X_1 = x_1, \ldots, X_p = x_p ) } = \frac{  P( X_1 = x_1, \ldots, X_p = x_p | Y=y ) P(Y=y) } { P( X_1 = x_1, \ldots, X_p = x_p ) } $$
Note that $y$ appears only in the numerator, so maximizing the posterior is equivalent to maximizing the numerator, which is essentially a re-expression of the *joint* distribution. This is the form of the maximization we will perform, in order to avoid unnecessary calculations.

Also note that *if* the predictor RVs $X_1,\ldots, X_p$ were *independent* of the response RV $Y$, then its conditional/posterior distribution would be the same as its *marginal* distribution:
$$P( Y = y | X_1 =x_1, \ldots, X_p = x_p ) = P( Y = y)$$
Intuitively, independence implies that the information obtained from $X_1, \ldots, X_p$ is irrelevant for predicting $Y$. When building a classifier, we therefore want to use predictors that are as *dependent* on the response as possible. As an extreme example, if you had a predictor $X$ that was *perfectly* related to $Y$, i.e. there is a 1-to-1 relation between the values of $X$ and $Y$, then knowing the value of $X$ would tell you the value of $Y$.

##### Naive Bayes Assumption

We have seen how Bayesian classification works, at least conceptually, but to actually *apply* it we need a workable form of the *joint* distribution of $Y,X_1,\ldots, X_p$. Since all variables are finite, their joint distribution can be represented as a $(1+p)$-dimensional array. In Statistical/Machine Learning applications the probability model is not provided, but has to be *estimated/trained* based on data. E.g., assuming all variables are *binary*, there are $2^{p+1}-1$ free probabilities to estimate in the joint distribution array. Because the number of parameters increases *exponentially* in the number of dimensions/predictors, good estimation quickly becomes infeasible (it requires astronomical amounts of data); this is known as the [curse of dimensionality](https://en.wikipedia.org/wiki/Curse_of_dimensionality).

To solve this problem, NBC makes a *naive* (not realistic) simplifying assumption: it assumes the features are *conditionally independent* given the class $Y$. In practice, this implies that:
$$ P( X_1 = x_1, \ldots, X_p = x_p | Y=y ) P(Y=y)
= \left( \prod_{i=1}^{p} P(X_i = x_i | Y = y ) \right)  \times P( Y = y ) $$
I.e., the joint distribution can be expressed as a product of the $X$'s 1D conditionals times $Y$'s 1D marginal. By imposing this special structure we effectively reduce the dimensionality of the problem: instead of one $(p+1)$-dimensional distribution, we work with $(p+1)$ 1-dimensional distributions. E.g., if we assume the RVs are binary, there are $2p+1$ probabilities to estimate ($P(Y=1)$, plus $P(X_i=1|Y=y)$ for each of the $p$ features and 2 classes), rather than $2^{p+1}-1$ for the general model. Now that we know the main idea behind NBC, we look at a specific application to spam detection.

##### Spam Data

We will be working with a subset of the [Enron-Spam dataset](https://auebgr-my.sharepoint.com/:f:/g/personal/nlp_aueb_gr/Em7LRLHD3p9OqKOB3oK8BXoBZMANfisgs-jg0VLsZGQGWw?e=vJdUIF) consisting of 5726 emails, 1368 of which (23.9%) are *spam* and the remaining 4358 *ham* (legitimate).
We use the [pandas](https://pandas.pydata.org/) library in Python to work with table/spreadsheet-like data, where each row represents an observation and each column a field/variable. Column ```text``` contains the email text, and column ```spam``` indicates if the email is spam (1) or not (0); below is a preview:

In [17]:
import pandas as pd
emails = pd.read_csv("./data/emails.csv", usecols=[0, 1])
emails.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


The ```spam``` column is the response, and the ```text``` will provide the features.

Before building anything, we split the data into a [training and test set](https://en.wikipedia.org/wiki/Training,_validation,_and_test_data_sets): the training set is used to *estimate* the probabilities, and the test set is used to *evaluate* the performance of the classifier. We do this to get an *objective* measure of performance on emails the classifier has not seen; measuring accuracy on the same data used for training would reward a classifier that merely memorizes them. We keep 25% of the emails for testing, and use the ```stratify``` option so that both sets have the same proportion of spam as the full data.

In [18]:
from sklearn.model_selection import train_test_split

text_train, text_test, Y_train, Y_test = train_test_split(
    emails.text, emails.spam, test_size = 0.25, random_state = 1234, stratify = emails.spam)

print( len(Y_train), "training emails,", len(Y_test), "test emails" )
print( "Spam proportion in training / test:", round(Y_train.mean(), 4), "/", round(Y_test.mean(), 4) )

4294 training emails, 1432 test emails
Spam proportion in training / test: 0.2389 / 0.2388


We construct binary predictor/feature variables that record the presence or absence of various words in each email. Looking at only the presence of words is a crude way to analyze text, disregarding its structure, and is called [bag-of-words](https://en.wikipedia.org/wiki/Bag-of-words_model) modelling.

To extract the word features, we use the function ```feature_extraction.text.CountVectorizer``` from ```scikit-learn```. The choice of word features relies on some heuristic rules: specifically, we remove English [stop-words](https://en.wikipedia.org/wiki/Stop_word) like articles or common verbs, as well as infrequent (<10 emails) or very frequent (>30% of emails) words, since they do not provide much information (e.g., if a word is in every/no email, it doesn't help distinguish spam). Each remaining word is represented by a *binary* feature where 1 means the word is present in the email and 0 otherwise. Note that the vocabulary is determined using the *training* emails only, and is then applied to the test emails. Below is the code to compute the features, the resulting number of words $p$, and a random sample of 10 feature words.

In [19]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words = 'english', binary = True, min_df = 10, max_df = .30)  # initialize vectorizer
X_train = vectorizer.fit_transform( text_train )  # learn vocabulary from training emails; output is a sparse binary matrix
X_test  = vectorizer.transform( text_test )       # apply the same vocabulary to test emails
X_words = vectorizer.get_feature_names_out()      # extract words

print( "Number of word features p =", len(X_words) )

import random; random.seed(1)
random.sample( list(X_words), 10 )

Number of word features p = 5220


['commodity',
 'supervisor',
 'apply',
 'francisco',
 'change',
 'requirements',
 'presenting',
 'racicot',
 'mi',
 'encouraged']

##### Probability Estimation

We now estimate the joint distribution, which for NBC is given by the conditional probabilities of each word being present in a spam or non-spam email, i.e. $P( X_i=1 | Y=1 )$ and $P( X_i = 1 | Y=0)$ respectively. Notice that for any binary feature we have $ P( X_i=0 | Y=y ) = 1 - P( X_i=1 | Y=y)$ for $y=0,1$ by the complement rule, i.e. just one probability value determines the conditional distribution.

Let's focus on $ P( X_i=1 | Y=1 ) = \frac{P( X_i=1, Y=1 ) }{P( Y=1 )}$: we could estimate this probability from data by dividing the number of spam emails containing word $i$ (corresponding to $ \{X_i=1\} \cap \{Y=1\}$) by the number of spam emails ($\{Y=1\}$) in our training data. But this approach has a problem: if no spam email in our data contains that word, the estimated probability will be 0, which implies it is *impossible* to ever have a spam email with that word. To avoid such extremes we employ [Laplace smoothing](https://en.wikipedia.org/wiki/Additive_smoothing), adjusting the ratio to allow a small probability of observing a word in every type of email. The estimated conditional probabilities become:

$$
\hat{P}( X_i=1 | Y=1 ) = \frac{ \text{ \# spam emails w/ word $i$}+1 }{ \text{ \# spam emails}+2 } \\
\hat{P}( X_i=1 | Y=0 ) = \frac{ \text{ \# ham emails w/ word $i$}+1 }{ \text{ \# ham emails}+2 } 
$$

(the *hat* notation $\hat{P}$ indicates this is an estimate). This, combined with the estimated (marginal) probability of spam:

$$ \hat{P}(Y=1) = \frac{ \text{ \# spam emails } } { \text{ \# emails } } $$

is all we need for NBC.

In [20]:
import numpy as np

n_spam = np.sum( Y_train == 1 );  n_ham = np.sum( Y_train == 0 )

# Number of training emails containing each word, separately for spam and ham:
# X_train[ (Y_train == 1).values ] uses Boolean indexing to extract the spam rows,
# and .sum(axis = 0) sums along the word columns.
count_spam = np.asarray( X_train[ (Y_train == 1).values ].sum(axis = 0) ).ravel()
count_ham  = np.asarray( X_train[ (Y_train == 0).values ].sum(axis = 0) ).ravel()

P_X_Y1 = ( count_spam + 1 ) / ( n_spam + 2 )   # Estimate P( X_i=1 | Y=1 )
P_X_Y0 = ( count_ham  + 1 ) / ( n_ham  + 2 )   # Estimate P( X_i=1 | Y=0 )
P_Y1 = n_spam / len(Y_train)                   # Estimate P( Y=1 )
P_Y0 = 1 - P_Y1

Before classifying, it is instructive to look at what the model has learned. The words with the largest *ratio* $\hat{P}( X_i=1 | Y=1 ) / \hat{P}( X_i=1 | Y=0 )$ are the ones most strongly associated with spam, and the smallest ratios identify words that signal legitimate email.

In [21]:
ratio = P_X_Y1 / P_X_Y0
order = np.argsort(ratio)

print("Most spam-like words:", list( X_words[ order[::-1][:10] ] ) )
print("Most ham-like words: ", list( X_words[ order[:10] ] ) )

Most spam-like words: ['viagra', 'stationery', 'sex', 'andmanyother', 'oniine', 'corel', 'wili', 'professionai', 'wiil', 'projecthoneypot']
Most ham-like words:  ['ect', 'hou', '713', 'crenshaw', 'stinson', 'gibner', 'shirley', '853', 'edu', 'vkamins']


##### Classification

With probabilities estimated, we now classify the emails. Using only the word features (the $X$'s), we compare the two quantities $P( X_1=x_1,\ldots,X_p=x_p | Y=y) P(Y=y)$ for $y=0$ (ham) and $y=1$ (spam), and choose the class with the highest one. By the Naive Bayes assumption this is factorized as:
$$ P( X_1 = x_1, \ldots, X_p = x_p | Y=y ) P(Y=y) =  \prod_{i=1}^{p} P(X_i = x_i | Y = y ) \times P( Y = y ) $$
If you tried to calculate this product in Python you would get 0, because it multiplies thousands of small probabilities (this is called *numerical underflow*). For numerical stability, we compare the *logarithms* of the products, which are the sums of their logs (the logarithm is increasing, so it does not change which class is larger):
$$ \log  \left( \prod_{i=1}^{p} P(X_i = x_i | Y = y ) \times P( Y = y ) \right) = \sum_{i=1}^{p} \log ( P(X_i = x_i | Y = y ) ) + \log(  P( Y = y ) ) $$
The following function computes these log-scores for *all* emails at once. Since each $x_i$ is either 1 or 0, we have $\log P(X_i = x_i | Y = y) = x_i \log P(X_i=1|Y=y) + (1-x_i) \log \left(1 - P(X_i=1|Y=y)\right)$, where we used the complement rule for words that are absent ($x_i=0$). Summing over $i$ then becomes a matrix-vector product with the feature matrix.

In [22]:
def nb_log_score(X, P_X_Y, P_Y):
    # log[ P(X = x | Y=y) P(Y=y) ] for every row x of the binary feature matrix X
    X = X.toarray()      # sparse -> dense array (small enough for this data)
    return X @ np.log(P_X_Y) + (1 - X) @ np.log(1 - P_X_Y) + np.log(P_Y)

def nb_classify(X):
    lP_Y0X = nb_log_score(X, P_X_Y0, P_Y0)
    lP_Y1X = nb_log_score(X, P_X_Y1, P_Y1)
    return (lP_Y1X > lP_Y0X).astype(int)         # choose spam (1) when it has the larger score

Y_my_pred = nb_classify(X_test)
np.mean( Y_my_pred == Y_test )                    # accuracy: proportion of correctly classified test emails

np.float64(0.9650837988826816)

The *accuracy*, i.e. the proportion of correctly classified emails, is an impressive 96.5% on emails the classifier has never seen. But accuracy alone can be misleading, since the two kinds of mistakes are not equally costly: sending a legitimate email to the spam folder (a *false positive*) is usually worse than letting a spam email through (a *false negative*). The [confusion matrix](https://en.wikipedia.org/wiki/Confusion_matrix) shows both: rows are the true classes (ham, spam) and columns the predicted ones.

In [23]:
from sklearn.metrics import confusion_matrix, classification_report
print( confusion_matrix(Y_test, Y_my_pred) )
print( classification_report(Y_test, Y_my_pred, target_names = ['ham', 'spam']) )

[[1042   48]
 [   2  340]]
              precision    recall  f1-score   support

         ham       1.00      0.96      0.98      1090
        spam       0.88      0.99      0.93       342

    accuracy                           0.97      1432
   macro avg       0.94      0.98      0.95      1432
weighted avg       0.97      0.97      0.97      1432



Here *precision* for spam is the proportion of emails flagged as spam that really are spam (estimating $P(Y=1|\hat{Y}=1)$), while *recall* is the proportion of spam emails that get flagged (estimating $P(\hat{Y}=1|Y=1)$).

We have effectively implemented the same procedure used in ```scikit-learn```, albeit in a more transparent way. For comparison, the following code trains ```BernoulliNB```, the NBC for binary features. Its smoothing parameter ```alpha=1``` corresponds exactly to the Laplace smoothing we used above, so the predictions should be identical.

In [24]:
from sklearn.naive_bayes import BernoulliNB

clf = BernoulliNB(alpha = 1)       # initiate Bernoulli (= binary feature) NBC, with Laplace smoothing
clf.fit(X_train, Y_train)          # train model
Y_pred = clf.predict(X_test)       # extract predictions

print( "Accuracy:", np.mean( Y_pred == Y_test ) )
print( "Same predictions as our implementation:", np.all( Y_pred == Y_my_pred ) )

Accuracy: 0.9650837988826816
Same predictions as our implementation: True


### Problems

1. Show that for a binary response $Y$ and discrete predictors $X_1, \ldots, X_p$, the NBC chooses spam ($Y=1$) exactly when the *log-odds* are positive:
$$ \log \frac{P(Y=1)}{P(Y=0)} + \sum_{i=1}^{p} \log \frac{P(X_i = x_i | Y = 1)}{P(X_i = x_i | Y = 0)} > 0 $$
Which words contribute positively to this sum, and which negatively?

2. The Laplace smoothing we used adds 1 to each count. Retrain the classifier with ```BernoulliNB(alpha = a)``` for $a = 10^{-10}, 10^{-3}, 1, 10$ and compare the test accuracy. What happens for tiny $a$ (essentially no smoothing), and why?

3. A 'false positive' (ham flagged as spam) is much worse than a false negative. Change the decision rule to flag an email as spam only if the posterior probability $P(Y=1|x_1,\ldots,x_p)$ exceeds a threshold $t$ (use ```clf.predict_proba```). Compute the false positive and false negative counts on the test set for $t=0.5, 0.9, 0.99, 0.999$.
4. Suppose you add a new feature that is an exact *duplicate* of an existing, highly spam-indicative word. How does this affect the Naive Bayes classification, and which modelling assumption does it violate?

<a rel="license" href="http://creativecommons.org/licenses/by-nc-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-nc-sa/4.0/88x31.png" /></a><br />This work is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by-nc-sa/4.0/">Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International License</a>.